#### 汇总md正文内容为一个连续的md文件

In [1]:
import re
from pathlib import Path

class MarkdownMainTextGatherer:
    def __init__(self, md_root_dir, output_file):
        self.md_root_dir = Path(md_root_dir)
        self.output_file = Path(output_file)
        self.fig_table_pattern = re.compile(r'^(图|附图|附表|表)\s?\d+.*')
        self.image_link_pattern = re.compile(r'^!\[.*\]\(.*\)')
        self.page_divider_pattern = re.compile(r'^\{\d+\}-+')
        self.metadata_keywords = ["批次", "处理时间", "总字符数", "---"]
        self.list_line_pattern = re.compile(r'^（\d+）')

    def find_md_files(self) -> list:
        """查找符合命名模式的md文件并排序"""
        md_files = []
        pattern = re.compile(r'batch\d+_page\d+-\d+_clean\.md$')
        for file_path in self.md_root_dir.rglob('*.md'):
            if pattern.search(file_path.name):
                md_files.append(file_path)
        return sorted(md_files)

    def normalize_text(self, text: str) -> str:
        return text.strip()

    def is_valid_line(self, line: str) -> bool:
        line = line.strip()
        if not line:
            return False
        if any(keyword in line for keyword in self.metadata_keywords):
            return False
        if self.page_divider_pattern.match(line):
            return False
        return True

    def is_heading(self, line: str) -> bool:
        return line.strip().startswith('#')

    def is_fig_caption(self, line: str) -> bool:
        return bool(self.fig_table_pattern.match(line.strip()))

    def is_image(self, line: str) -> bool:
        return bool(self.image_link_pattern.match(line.strip()))

    def is_list_line(self, line: str) -> bool:
        s = line.strip()
        if s.startswith('•'):
            return True
        if self.list_line_pattern.match(s):
            return True
        if s.startswith('-') and len(s) > 1 and s[1] in ' \t（(0123456789':
            return True
        return False

    def is_sentence_end(self, line: str) -> bool:
        return line.endswith(('。', '！', '？', '”', '」', '』', '；', '.', '!', '?', '"'))

    def process(self):
        md_files = self.find_md_files()
        print(f"找到 {len(md_files)} 个匹配的 Markdown 文件。")

        all_content_blocks = []
        for file_path in md_files:
            with open(file_path, 'r', encoding='utf-8') as f:
                for line in f:
                    clean_line = line.strip()
                    if self.is_valid_line(clean_line):
                        all_content_blocks.append(clean_line)

        final_lines = []
        if not all_content_blocks:
            print("没有找到有效内容。")
            return

        blocks = all_content_blocks
        n = len(blocks)
        i = 0
        while i < n:
            line = blocks[i]
            if self.is_heading(line):
                final_lines.append(line)
                i += 1
                continue
            if self.is_fig_caption(line) or self.is_image(line):
                final_lines.append(line)
                i += 1
                continue
            if self.is_list_line(line):
                final_lines.append(line)
                i += 1
                continue

            merged = line
            pending_figs = []
            pos = i + 1
            while not self.is_sentence_end(merged):
                while pos < n and (self.is_fig_caption(blocks[pos]) or self.is_image(blocks[pos])):
                    pending_figs.append(blocks[pos])
                    pos += 1
                if pos >= n:
                    break
                nxt = blocks[pos]
                if self.is_heading(nxt) or self.is_list_line(nxt):
                    break
                merged = merged + nxt
                pos += 1

            final_lines.append(merged)
            final_lines.extend(pending_figs)
            i = pos

        with open(self.output_file, 'w', encoding='utf-8') as f:
            full_text = "\n\n".join([l for l in final_lines if l.strip()])
            f.write(full_text)

        print(f"汇总完成！文件已保存至: {self.output_file}")

# --- Jupyter Cell 运行示例 ---
md_root = r"knowledgeBase\pdfParse\cleaned_data"
output_md = r"knowledgeBase\pdfParse\cleaned_data\cleaning\consolidated_corpus_3.md"

gatherer = MarkdownMainTextGatherer(md_root, output_md)
gatherer.process()

找到 17 个匹配的 Markdown 文件。
汇总完成！文件已保存至: knowledgeBase\pdfParse\cleaned_data\cleaning\consolidated_corpus_3.md


### 将md切分为与